# Valley Fever | Controlled environmental-feature experiment

**Checkpoint 2 follow-up:** Does adding PM10, rainfall and AZMet weather actually improve predictions beyond case history?

This notebook continues the existing three-county project. It does **not** collect or rebuild the original datasets. It uses the exact `VF_ThreeCounty_Stage2_Model_Input.csv` already in the project ZIP.

- **Case-only:** current cases, same month last year, seasonal month sine/cosine.
- **Case + environment:** identical model plus PRISM rainfall, EPA PM10, AZMet soil20, humidity, wind and VPD.
- **Models:** NB2 for Maricopa; pooled `XGB_medium` for Pinal and Pima (frozen from earlier development selections).
- **Evaluation:** expanding-year 2019–2023, retrospective 2024, provisional 2025, and January–March 2024 sensitivity.

**Important:** No 2024 outcome is used to fit 2024 models, but the 2024 test period was previously inspected during this project. Previous-month case counts may not have been available at actual forecast time.

In [ ]:
# Step 1: Find or upload the exact combined Tier 2 CSV.
import os, importlib.util, subprocess, sys
from pathlib import Path
required={'numpy':'numpy','pandas':'pandas','sklearn':'scikit-learn','xgboost':'xgboost','statsmodels':'statsmodels','matplotlib':'matplotlib'}
missing=[pip for mod,pip in required.items() if importlib.util.find_spec(mod) is None]
if missing:subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
source=Path('VF_ThreeCounty_Stage2_Model_Input.csv')
if not source.exists():
    try:
        from google.colab import files
        print('Upload VF_ThreeCounty_Stage2_Model_Input.csv (included in the ZIP).')
        uploaded=files.upload()
        if source.name not in uploaded:raise FileNotFoundError(f'Expected {source.name}')
    except ImportError:
        raise FileNotFoundError(f'Place {source.name} in the notebook working folder.')
os.environ['VF_INPUT']=str(source)
os.environ['VF_OUT']='VF_Environment_Ablation_Results'
print('Input:',source,'bytes:',source.stat().st_size)

## Step 2: Fixed features and controlled model fitting
Both arms use identical training and test months. This is an ablation, not a new model-selection search.

In [ ]:
"""Controlled Valley Fever case-history versus environment ablation.
Input: exact 357-row combined Tier 2 CSV; fixed model families from previously
selected development models; same targets/rows/training windows in each pair.
This does not claim a never-seen 2024 test, real-time case availability, or causality.
"""
import warnings, os, json, sys
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor
import statsmodels.api as sm
from statsmodels.discrete.discrete_model import NegativeBinomial
import xgboost, sklearn, statsmodels

SOURCE = Path(os.environ.get('VF_INPUT', 'VF_ThreeCounty_Stage2_Model_Input.csv'))
OUT = Path(os.environ.get('VF_OUT', 'VF_Environment_Ablation_Results'))
OUT.mkdir(parents=True, exist_ok=True)
SEED=42
COUNTIES=['Maricopa','Pinal','Pima']
XGB_PARAMS=dict(n_estimators=190,max_depth=3,learning_rate=0.04,min_child_weight=5,subsample=0.80,colsample_bytree=0.80,reg_lambda=15)
ENV_FEATURES=['rain_log','pm10','soil20','rh','wind','vpd']
CASE_FEATURES=['case_log','seasonal_log','month_sin','month_cos']
EXTRA_CASE_FEATURES=['case_lag1_log','case_lag2_log']


def load():
    d=pd.read_csv(SOURCE,dtype={'County_FIPS':str})
    d['target_dt']=pd.to_datetime(d.Target_Month,format='%Y-%m')
    d['feature_dt']=pd.to_datetime(d.Feature_Month,format='%Y-%m')
    d['Year']=d.target_dt.dt.year.astype(int)
    d['Month']=d.target_dt.dt.month.astype(int)
    assert len(d)==357 and d.groupby('County').size().to_dict()=={'Maricopa':119,'Pima':119,'Pinal':119}
    assert not d.duplicated(['County','Target_Month']).any()
    assert ((d.feature_dt+pd.offsets.MonthBegin(1))==d.target_dt).all()
    assert set(d.Year)==set(range(2016,2026))
    for col in ['Target_Cases_Next_Month','VF_cases_current','VF_cases_lag1','VF_cases_lag2','Seasonal_Baseline_Cases']:
        assert pd.to_numeric(d[col],errors='coerce').notna().all(),col
    for col in ['PRISM_rain_total_in_current','EPA_PM10_mean_ugm3_current','AZMET_soil20_C_current','AZMET_RH_pct_current','AZMET_wind_mean_ms_current','AZMET_VPD_kPa_current']:
        assert pd.to_numeric(d[col],errors='coerce').notna().all(),col
    assert dict(d[d.Year==2024].groupby('County').Target_Cases_Next_Month.sum())=={'Maricopa':10920,'Pima':1614,'Pinal':1484}
    return d.sort_values(['Year','Month','County']).reset_index(drop=True)


def make_x(d, environmental, strong_cases, pooled):
    z=pd.DataFrame(index=d.index)
    z['case_log']=np.log1p(d.VF_cases_current.to_numpy(dtype=float))
    z['seasonal_log']=np.log1p(d.Seasonal_Baseline_Cases.to_numpy(dtype=float))
    z['month_sin']=np.sin(2*np.pi*d.Month/12)
    z['month_cos']=np.cos(2*np.pi*d.Month/12)
    if strong_cases:
        z['case_lag1_log']=np.log1p(d.VF_cases_lag1.to_numpy(dtype=float))
        z['case_lag2_log']=np.log1p(d.VF_cases_lag2.to_numpy(dtype=float))
    if environmental:
        z['rain_log']=np.log1p(d.PRISM_rain_total_in_current.to_numpy(dtype=float))
        z['pm10']=d.EPA_PM10_mean_ugm3_current.to_numpy(dtype=float)
        z['soil20']=d.AZMET_soil20_C_current.to_numpy(dtype=float)
        z['rh']=d.AZMET_RH_pct_current.to_numpy(dtype=float)
        z['wind']=d.AZMET_wind_mean_ms_current.to_numpy(dtype=float)
        z['vpd']=d.AZMET_VPD_kPa_current.to_numpy(dtype=float)
    if pooled:
        z['is_maricopa']=(d.County=='Maricopa').astype(float)
        z['is_pinal']=(d.County=='Pinal').astype(float)
    assert np.isfinite(z.to_numpy()).all()
    return z


def predict(tr,te,county,environmental,strong_cases):
    # Identical model family, tuning and rows within each paired comparison.
    pooled=(county!='Maricopa')
    Xtr=make_x(tr,environmental,strong_cases,pooled)
    Xte=make_x(te,environmental,strong_cases,pooled)
    y=tr.Target_Cases_Next_Month.to_numpy(dtype=float)
    if county=='Maricopa':
        sc=StandardScaler()
        a=sm.add_constant(sc.fit_transform(Xtr),has_constant='add')
        b=sm.add_constant(sc.transform(Xte),has_constant='add')
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            fit=NegativeBinomial(y,a,loglike_method='nb2').fit(method='bfgs',maxiter=170,disp=0)
        if not bool(fit.mle_retvals.get('converged',False)):
            raise RuntimeError(f'NB2 did not converge: {county} env={environmental} strong={strong_cases}')
        p=fit.predict(b)
        model='NB2 county-only'
    else:
        reg=XGBRegressor(objective='reg:squarederror',tree_method='hist',random_state=SEED,n_jobs=1,verbosity=0,**XGB_PARAMS)
        reg.fit(Xtr,np.log1p(y))
        p=np.expm1(reg.predict(Xte))
        model='XGB_medium pooled'
    if not np.isfinite(p).all():raise RuntimeError('Nonfinite predictions')
    return np.maximum(0,np.asarray(p,dtype=float)),model


def run():
    d=load()
    rows=[]
    meta=[]
    for strong_cases in [False,True]:
        specification='ORIGINAL_CASE_FEATURES' if not strong_cases else 'STRONGER_CASE_HISTORY'
        for phase in ['DEV_2019_2023','TEST_2024','PROVISIONAL_2025']:
            years=[2019,2020,2021,2022,2023] if phase=='DEV_2019_2023' else ([2024] if phase=='TEST_2024' else [2025])
            for year in years:
                for environmental in [False,True]:
                    for county in COUNTIES:
                        pooled=county!='Maricopa'
                        tr=d[(d.Year<year) & (pd.Series(True,index=d.index) if pooled else (d.County==county))]
                        te=d[(d.Year==year)&(d.County==county)]
                        assert len(te)==12 and not tr.empty
                        p,model=predict(tr,te,county,environmental,strong_cases)
                        group='CASE_PLUS_ENV' if environmental else 'CASE_ONLY'
                        meta.append(dict(specification=specification,phase=phase,year=year,county=county,model=model,feature_group=group,train_rows=len(tr),test_rows=len(te),train_max_year=int(tr.Year.max())))
                        for (_,r),pred in zip(te.iterrows(),p):
                            actual=float(r.Target_Cases_Next_Month)
                            rows.append(dict(specification=specification,phase=phase,county=county,year=year,target_month=r.Target_Month,model=model,feature_group=group,
                                actual=actual,predicted=float(pred),absolute_error=float(abs(actual-pred)),
                                seasonal_pred=float(r.Seasonal_Baseline_Cases),previous_month_pred=float(r.VF_cases_current),
                                early_2024_flag=r.Early_2024_Surveillance_Flag))
    pred=pd.DataFrame(rows)
    assert len(pred)==2*2*(5+1+1)*3*12
    assert pred.groupby(['specification','phase','county','target_month']).feature_group.nunique().eq(2).all()
    rec=[]
    for (spec,phase,county,feature_group),ss in pred.groupby(['specification','phase','county','feature_group']):
        for period in ['All','Apr_Dec','Jan_Mar']:
            z=ss if period=='All' else ss[ss.target_month.str[-2:].astype(int).ge(4) if period=='Apr_Dec' else ss.target_month.str[-2:].astype(int).le(3)]
            # For development period, Apr-Dec means 9*5 months, not 9.
            actual=z.actual.to_numpy(); p=z.predicted.to_numpy()
            rec.append(dict(specification=spec,phase=phase,county=county,model=ss.model.iloc[0],feature_group=feature_group,period=period,
                N_months=len(z),MAE=float(np.mean(np.abs(actual-p))),RMSE=float(np.sqrt(np.mean((actual-p)**2))),
                WAPE_pct=float(100*np.sum(np.abs(actual-p))/np.sum(actual)),
                seasonal_MAE=float(np.mean(np.abs(actual-z.seasonal_pred.to_numpy()))),
                previous_month_MAE=float(np.mean(np.abs(actual-z.previous_month_pred.to_numpy())))))
    metric=pd.DataFrame(rec)
    wide=metric.pivot(index=['specification','phase','county','model','period'],columns='feature_group',values='MAE').reset_index()
    wide.columns.name=None
    wide['ENV_GAIN_CASES']=wide['CASE_ONLY']-wide['CASE_PLUS_ENV']
    wide['ENV_GAIN_PCT']=100*wide.ENV_GAIN_CASES/wide.CASE_ONLY
    wide['ENV_HELPED']=np.where(wide.ENV_GAIN_CASES>0,'YES','NO')
    extra=metric[metric.feature_group=='CASE_ONLY'][['specification','phase','county','period','N_months','seasonal_MAE','previous_month_MAE']]
    wide=wide.merge(extra,on=['specification','phase','county','period'],validate='one_to_one')
    wide=wide.sort_values(['specification','phase','county','period']).reset_index(drop=True)
    pred.to_csv(OUT/'Monthly_Paired_Predictions.csv',index=False)
    metric.to_csv(OUT/'All_Model_Metrics.csv',index=False)
    wide.to_csv(OUT/'Paired_Comparison.csv',index=False)
    pd.DataFrame(meta).to_csv(OUT/'Training_Audit.csv',index=False)
    versions={'python':sys.version.split()[0],'numpy':np.__version__,'pandas':pd.__version__,'xgboost':xgboost.__version__,'sklearn':sklearn.__version__,'statsmodels':statsmodels.__version__}
    (OUT/'Versions.json').write_text(json.dumps(versions,indent=2))
    # Extra robustness only; 2025 remains provisional and previously inspected.
    q1_rows=[]
    for strong in [False,True]:
        spec='ORIGINAL_CASE_FEATURES' if not strong else 'STRONGER_CASE_HISTORY'
        for county in COUNTIES:
            pooled=county!='Maricopa'
            train=d[(d.Year<2025) & ((d.County==county) if not pooled else pd.Series(True,index=d.index))]
            train=train[~((train.Year==2024)&(train.Month<=3))]
            test=d[(d.Year==2025)&(d.County==county)]
            for env in [False,True]:
                p,model=predict(train,test,county,env,strong)
                q1_rows.append(dict(specification=spec,county=county,model=model,training='2024_JAN_MAR_EXCLUDED',feature_group='CASE_PLUS_ENV' if env else 'CASE_ONLY',N_months=len(test),MAE=float(np.mean(np.abs(test.Target_Cases_Next_Month.to_numpy()-p)))))
    q1=pd.DataFrame(q1_rows)
    q1.to_csv(OUT/'2025_Q1_Training_Sensitivity.csv',index=False)
    # How often environmental features reduce monthly absolute error (not a significance test).
    monthly24=pred[(pred.specification=='ORIGINAL_CASE_FEATURES')&(pred.phase=='TEST_2024')].pivot(index=['county','target_month'],columns='feature_group',values='absolute_error').reset_index()
    monthly24['env_reduced_error']=monthly24.CASE_PLUS_ENV<monthly24.CASE_ONLY
    monthwins=monthly24.groupby('county',as_index=False).agg(months_environment_better=('env_reduced_error','sum'),months_total=('env_reduced_error','size'))
    monthwins.to_csv(OUT/'2024_Monthly_Win_Counts.csv',index=False)
    print('VERSIONS:',versions)
    for spec in ['ORIGINAL_CASE_FEATURES','STRONGER_CASE_HISTORY']:
        print('\n',spec)
        for phase in ['DEV_2019_2023','TEST_2024','PROVISIONAL_2025']:
            print(phase)
            print(wide[(wide.specification==spec)&(wide.phase==phase)&(wide.period=='All')][['county','CASE_ONLY','CASE_PLUS_ENV','ENV_GAIN_CASES','ENV_GAIN_PCT','ENV_HELPED','seasonal_MAE','previous_month_MAE']].round(2).to_string(index=False))
        print('APR-DEC 2024')
        print(wide[(wide.specification==spec)&(wide.phase=='TEST_2024')&(wide.period=='Apr_Dec')][['county','CASE_ONLY','CASE_PLUS_ENV','ENV_GAIN_CASES','ENV_GAIN_PCT','ENV_HELPED']].round(2).to_string(index=False))
    return pred,metric,wide




## Step 3: Run all comparisons
Results include monthly paired predictions, full/Apr–Dec/Jan–Mar metrics, a stronger-case-history check and 2025 training sensitivity.

In [ ]:
pred,metrics,comparison=run()
print('Saved output files:',*[p.name for p in sorted(OUT.glob('*'))],sep='\n- ')

## Step 4: Save a single results ZIP
Download the CSVs and model versions. Compare with the provided Excel summary.

In [ ]:
import zipfile
zip_name='VF_Environment_Ablation_Colab_Results.zip'
with zipfile.ZipFile(zip_name,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.iterdir()):
        if p.is_file():z.write(p,arcname=p.name)
assert zipfile.ZipFile(zip_name).testzip() is None
print('Ready:',zip_name)
try:
    from google.colab import files
    files.download(zip_name)
except ImportError:
    print('Find ZIP in working directory.')